# Define our functions

In [1]:
import math
import matplotlib.pyplot as plt
import glob
import cv2
import numpy as np
import scipy.constants as sc
from sklearn.cluster import KMeans
from skimage import measure
from skimage.morphology import skeletonize
import random
from collections import deque
import os
import PIL
from scipy.fft import fft2, ifft2, fftshift
from joblib import Parallel, delayed
from PIL import Image
from PIL import ImageEnhance
from PIL import ImageFilter  
from skimage import io
from cupy_common import check_cupy_available
# from PIL import ImageDraw
gpu_accelerated = check_cupy_available()

In [2]:
def compute_average_angle(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    blurred_img = cv2.GaussianBlur(image, (3, 3), 0)
    # skeleton = skeletonize(blurred_img // 255, method='lee').astype(np.uint8) * 255
    skeleton = skeletonize(blurred_img // 255, method='lee')

    # force to real ndarray
    skeleton_uint8 = np.array(skeleton, dtype=np.uint8, copy=True) * 255
    skeleton_uint8 = np.ascontiguousarray(skeleton_uint8)
    
    # print("IS EXACT BASE NDARRAY:", type(skeleton_uint8) is np.ndarray)
    # print("MRO:", skeleton_uint8.__class__.__mro__)
    # print(skeleton_uint8)
    _, binary_skeleton = cv2.threshold(skeleton_uint8, 127, 255, cv2.THRESH_BINARY)

    _, binary_skeleton = cv2.threshold(skeleton, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_skeleton, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    angles = []
    for contour in contours:
        for i in range(0, len(contour) - 10, 10):
            subcontour = contour[i:i+10]
            if len(subcontour) >= 2:
                dx = subcontour[-1][0][0] - subcontour[0][0][0]
                dy = subcontour[-1][0][1] - subcontour[0][0][1]
                angles.append(math.degrees(math.atan2(dy, dx)))
    
    return sum(angles) / len(angles) if angles else 0

In [3]:
class ColorWheelProcessor:
    def __init__(self, binarized_image, gpu_accelerated, color_wheel_origin=0):
        self.binarized_image = binarized_image
        self.sym = 2
        self.color = 5
        self.brightness = 1
        self.contrast = 5
        self.gpu_accelerated = gpu_accelerated
        self.color_wheel_origin = math.radians(color_wheel_origin)  # Convert degrees to radians

    def process_image(self):
        data = np.array(self.binarized_image)
        clrwhl = self._bldclrwhl(data.shape[0], data.shape[1], self.sym)
        imnp = self._nofft(clrwhl, data, data.shape[1], data.shape[1])
        imnp = imnp - np.min(imnp)
        imnp = imnp / np.max(imnp) * 255
        rgb2 = Image.fromarray(np.uint8(imnp))
        img2 = rgb2.filter(ImageFilter.GaussianBlur(radius=0.5))
        converter = ImageEnhance.Color(img2)
        img2 = converter.enhance(self.color)
        converter = ImageEnhance.Brightness(img2)
        img2 = converter.enhance(self.brightness)
        converter = ImageEnhance.Contrast(img2)
        img2 = converter.enhance(self.contrast)
        return img2

    def _bldclrwhl(self, nx, ny, sym):
        cda = cp.ones((nx, ny, 2))
        cx = cp.linspace(-nx, nx, nx)
        cy = cp.linspace(-ny, ny, ny)
        cxx, cyy = cp.meshgrid(cy, cx)
        
        # Apply the color wheel origin offset
        czz = (((cp.arctan2(cxx, cyy) - self.color_wheel_origin) / math.pi + 1.0) / 2.0) * sym
        cd2 = cp.dstack((czz, cda))
        carr = cd2
        chi = cp.floor(carr[..., 0] * 6)
        f = carr[..., 0] * 6 - chi
        p = carr[..., 2] * (1 - carr[..., 1])
        q = carr[..., 2] * (1 - f * carr[..., 1])
        t = carr[..., 2] * (1 - (1 - f) * carr[..., 1])
        v = carr[..., 2]
        chi = cp.stack([chi, chi, chi], axis=-1).astype(cp.uint8) % 6
        out = cp.choose(
            chi, cp.stack([cp.stack((v, t, p), axis=-1),
                           cp.stack((q, v, p), axis=-1),
                           cp.stack((p, v, t), axis=-1),
                           cp.stack((p, q, v), axis=-1),
                           cp.stack((t, p, v), axis=-1),
                           cp.stack((v, p, q), axis=-1)]))
        if self.gpu_accelerated:
            return cp.asnumpy(out)
        else:
            return out

    def _nofft(self, whl, img, nx, ny):
        imnp = cp.array(img)
        fimg = cp.fft.fft2(imnp)
        whl = cp.fft.fftshift(whl)
        proimg = cp.zeros((nx, ny, 3))
        comb = cp.zeros((nx, ny, 3), dtype=complex)
        magnitude = cp.repeat(np.abs(fimg)[:, :, np.newaxis], 3, axis=2)
        phase = cp.repeat(np.angle(fimg)[:, :, np.newaxis], 3, axis=2)
        proimg = whl * magnitude
        comb = cp.multiply(proimg, cp.exp(1j * phase))
        for n in range(3):
            proimg[:, :, n] = cp.real(cp.fft.ifft2(comb[:, :, n]))
            proimg[:, :, n] = proimg[:, :, n] - cp.min(proimg[:, :, n])
            proimg[:, :, n] = proimg[:, :, n] / cp.max(proimg[:, :, n])

        if self.gpu_accelerated:
            return cp.asnumpy(proimg)
        else:
            return proimg


In [4]:
# Inputs the color wheel image, subtracts the binarized image from it resulting in a colored, one phase image
class PhaseSubtraction:
    def __init__(self, input_image, binarized_image):
        self.input_image = input_image
        self.binarized_image = binarized_image

    def subtract_black_from_input(self):
        # Convert images to NumPy arrays
        input_array = np.array(self.input_image)
        binned_array = np.array(self.binarized_image)

        # Ensure the mask has the same number of channels as the input image
        if len(binned_array.shape) == 2:
            binned_array = np.expand_dims(binned_array, axis=-1)

        # Subtract black parts of the mask from the input image
        result_array = np.where(binned_array == 0, input_array, 255)

        # Create a PIL Image from the result array
        result_image = Image.fromarray(result_array.astype(np.uint8))
        return result_image

In [5]:
# Inputs the one phase image, creates masks for each orientation, outputs the filtered masks
class ColorMaskProcessor:
    def __init__(self, input_image, output_path):
        self.input_image = input_image
        self.output_path = output_path

    def create_color_mask(self, num_clusters):
        # Convert the image to a NumPy array
        img_array = np.array(self.input_image)

        # Reshape the array to a list of RGB values
        reshaped_array = img_array.reshape((-1, 3))

        # Use k-means clustering to group similar colors
        kmeans = KMeans(n_clusters=num_clusters, random_state=42)
        kmeans.fit(reshaped_array)

        # Get the labels assigned to each pixel
        labels = kmeans.labels_

        # Reshape the labels back to the original image shape
        segmented_image = labels.reshape(img_array.shape[:2])

        # Create a mask for each cluster
        masks = [(segmented_image == i) for i in range(num_clusters)]

        return masks

    def save_masks_as_images(self, image, masks):
        # Save each mask as a separate image to the current sample folder
        for i, mask in enumerate(masks):
            color_mask = np.zeros_like(image)
            color_mask[mask] = image[mask]
            mask_image = Image.fromarray(color_mask)
            mask_image.save(os.path.join(self.output_path, f"mask_{i}.tiff"))
            
            # Identify non-black pixels
            mask_array = np.array(mask_image)
            non_black_pixels = (mask_array[:, :, :3] > 0).any(axis=2)

            # Remove small clusters
            non_black_pixels = self.remove_small_clusters(non_black_pixels, min_size=15)

            # Create a new image with the modified non-black pixels
            result_img_array = np.zeros_like(mask_array)
            result_img_array[non_black_pixels] = mask_array[non_black_pixels]
            
            result_img = Image.fromarray(result_img_array, self.input_image.mode)
            result_img.save(os.path.join(self.output_path, f"filtered_mask_{i}.tiff"))


    def remove_small_clusters(self, image, min_size):
        labeled_image, num_labels = measure.label(image, connectivity=2, return_num=True)
        for label in range(1, num_labels + 1):
            cluster_size = np.sum(labeled_image == label)
            if cluster_size < min_size:
                image[labeled_image == label] = 0  # Set pixels in the small cluster to black
        return image

    def process_image(self):
        # Create color masks
        masks = self.create_color_mask(num_clusters=4)

        # Save each mask as a separate image
        image = self.input_image
        self.save_masks_as_images(np.array(image), masks)

In [64]:
from collections import deque
from PIL import Image
import random
import os
import numpy as np

class GrainFinder:
    def __init__(self, mask, directory, global_avg, save=False):
        self.mask = mask
        self.save = save
        self.directory = directory
        self.global_avg = global_avg

        self.original_image = Image.open(f'./{self.directory}/filtered_mask_{self.mask}.tiff')
        self.width, self.height = self.original_image.size
        self.original_pixels = np.array(self.original_image)

        # Needed by your outer pipeline
        self.group_sizes = {}

    # ---------------------------------------------------------
    # FLOOD-FILL TO IDENTIFY GROUPS
    # ---------------------------------------------------------
    def group_pixels(self, label_array, x, y, gid):
        queue = deque([(x, y)])
        label_array[y, x] = gid
        size = 1

        while queue:
            cx, cy = queue.popleft()

            for dx in range(-8, 9):
                for dy in range(-8, 9):
                    nx, ny = cx + dx, cy + dy
                    if (
                        0 <= nx < self.width and
                        0 <= ny < self.height and
                        label_array[ny, nx] == 0 and
                        not np.array_equal(self.original_pixels[ny, nx], [0, 0, 0])
                    ):
                        label_array[ny, nx] = gid
                        size += 1
                        queue.append((nx, ny))

        return size

    # ---------------------------------------------------------
    # MAIN PROCESSING
    # ---------------------------------------------------------
    def process_image(self):
        thresholds = np.linspace(0, 3.0, 15)
        results = []

        # ----------------------------------------
        # FIRST STAGE: Identify grain groups ONCE
        # ----------------------------------------
        label_array = np.zeros((self.height, self.width), dtype=np.int32)
        self.group_sizes = {}       # <-- now persistent
        gid = 1

        for y in range(self.height):
            for x in range(self.width):
                if (
                    not np.array_equal(self.original_pixels[y, x], [0, 0, 0])
                    and label_array[y, x] == 0
                ):
                    size = self.group_pixels(label_array, x, y, gid)
                    self.group_sizes[gid] = size
                    gid += 1

        # Save this once so threshold loops start from identical base
        base_labels = label_array.copy()

        # ----------------------------------------
        # THRESHOLD LOOP
        #----------------------------------------
        for thresh in thresholds:
            thresh_dir = f'./{self.directory}/{thresh:.1f}'
            os.makedirs(thresh_dir, exist_ok=True)

            # Reset to original labels
            label_array = base_labels.copy()

            keep_cutoff = self.global_avg * thresh
            kept = {g: s for g, s in self.group_sizes.items() if s >= keep_cutoff}
            filtered_ids = set(self.group_sizes.keys()) - set(kept.keys())

            results.append((thresh, list(kept.values())))

            # Remove filtered groups
            for g in filtered_ids:
                label_array[label_array == g] = 0

            # ----------------------------------------
            # Build colored visualization
            # ----------------------------------------
            rgb = np.zeros((self.height, self.width, 3), dtype=np.uint8)

            unique_gids = sorted([g for g in np.unique(label_array) if g != 0])
            color_map = {
                g: [random.randint(0, 255), random.randint(0, 255), random.randint(0, 255)]
                for g in unique_gids
            }

            for g, color in color_map.items():
                rgb[label_array == g] = color

            # Save outputs
            if self.save:
                out_img = Image.fromarray(rgb)
                out_img.save(f"{thresh_dir}/grains_{self.mask}_thresh_{thresh:.1f}.png")

                with open(f"{thresh_dir}/grains_{self.mask}_thresh_{thresh:.1f}.txt", "w") as f:
                    for g, s in kept.items():
                        f.write(f"Group {g}: {s} pixels\n")

        return results

In [65]:
%%time

# Here we will call our classes and loop through our samples
# If you need to change the directory addresses entirely, they are refrenced in cells 2 & 6.

if gpu_accelerated:
    print("Running on GPU")
    cp = __import__("cupy")
    gpu_accel=True
else:
    print("Running on CPU")
    gpu_accel=False
    cp = __import__("numpy")


# Number related to your fist sample
sample = 1
input_folder = 'data_set//'
# List of desired extensions
# extensions = ["*.tiff", "*.png"]

extensions = ["*.tiff"]
# extensions = ["B*", "C*", "D*", "E*", "F*", "X*", "Z*"]


# Find all files with matching extensions
image_files = [f for ext in extensions for f in glob.glob(os.path.join(input_folder, ext))]

#Create a loop for all samples
for image_file in image_files:

    # Extract the base name of the file for output folder naming
    image_name = os.path.splitext(os.path.basename(image_file))[0]
    print(f"Processing {image_name}")
    # output_folder = f'output_reviewer_onemask_v2/{image_name}'
    output_folder = f'final_threshold_output_v1/{image_name}'

    # Read in the image
    image = cv2.imread(image_file, cv2.IMREAD_GRAYSCALE).astype(np.uint8) * 255
    
    orientation_angle = compute_average_angle(image_file)

    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Create an instance of ColorWheel
    processor = ColorWheelProcessor(image, gpu_accel, color_wheel_origin=orientation_angle)
    # Call the instance to run the class
    processed_img = processor.process_image()

    # Create an instance of PhaseSubtraction
    phase_sub = PhaseSubtraction(processed_img, image)
    one_phase = phase_sub.subtract_black_from_input()

    # Create an instance of ColorMaskProcessor
    mask_maker = ColorMaskProcessor(one_phase, output_folder)
    mask_maker.process_image()

    mask_counts = []
    for i in range(4):
        img = Image.open(f"{output_folder}/filtered_mask_{i}.tiff")
        arr = np.array(img)
        non_black = np.count_nonzero(arr)
        mask_counts.append((i, non_black))
    
    ignore_mask = max(mask_counts, key=lambda x: x[1])[0]

    all_grain_sizes = []

    for i in range(4):
        if i == ignore_mask:
            continue
    
        tmp = GrainFinder(i, output_folder, global_avg=1, save=False)  # dummy
        tmp.process_image()  # but stop BEFORE thresholding
        all_grain_sizes.extend(tmp.group_sizes.values())
    
    global_avg = np.mean(all_grain_sizes)
    print(global_avg)

    # ---------------------------------------------------------
    # Now perform full thresholded runs and collect results
    # ---------------------------------------------------------
    combined_results = {}  # threshold → list of sizes across all masks
    
    for i in range(4):
        if i == ignore_mask:
            continue
    
        gf = GrainFinder(i, output_folder, global_avg, save=True)
        mask_results = gf.process_image()
    
        # Merge data from this mask
        for thresh, sizes in mask_results:
            combined_results.setdefault(thresh, []).extend(sizes)
    
    # ---------------------------------------------------------
    # Write ONE summary file per image
    # ---------------------------------------------------------
    summary_path = f"{output_folder}/grain_threshold_summary.txt"
    
    with open(summary_path, "w") as f:
        for thresh in sorted(combined_results):
            arr = np.array(combined_results[thresh], dtype=float)
            mean_sz = arr.mean() if arr.size else 0.0
            std_sz  = arr.std() if arr.size else 0.0
            f.write(f"{thresh:.1f}, {mean_sz:.3f}, {std_sz:.3f}\n")
    
    print(f"Saved unified summary → {summary_path}")

Running on CPU
Processing C_425


/home/bradley/anaconda3/envs/cangle/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


484.875
Saved unified summary → final_threshold_output_v1/C_425/grain_threshold_summary.txt
Processing D_450


/home/bradley/anaconda3/envs/cangle/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


726.275
Saved unified summary → final_threshold_output_v1/D_450/grain_threshold_summary.txt
Processing E_600


...

779.7333333333333
Saved unified summary → final_threshold_output_v1/B_525/grain_threshold_summary.txt
Processing B_550


/home/bradley/anaconda3/envs/cangle/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


433.4259259259259
Saved unified summary → final_threshold_output_v1/B_550/grain_threshold_summary.txt
Processing E_250


/home/bradley/anaconda3/envs/cangle/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


588.8913043478261
Saved unified summary → final_threshold_output_v1/E_250/grain_threshold_summary.txt
CPU times: user 1h 2min 1s, sys: 13.4 s, total: 1h 2min 14s
Wall time: 1h 1min 50s
